In [1]:
import os
import sys
os.path.dirname(sys.executable)

from pathlib import Path
from omegaconf import OmegaConf
import importlib
#from main_nolightning import initialize, configure
import pandas as pd
import numpy as np
from einops import rearrange
import torch
import pytorch_lightning as pl

%reload_ext autoreload
%autoreload 2
%matplotlib inline

# Dataloaders

In [2]:
from typing import Optional, List, Tuple
from abc import abstractmethod
from torch.utils.data import  DataLoader, RandomSampler, BatchSampler, SequentialSampler
        
from sklearn.preprocessing import MinMaxScaler

def check_boolean_vector(vec):
    # Find the index of the first False value
    try:
        first_false_index = np.where(vec == False)[0][0]
    except IndexError:  # If there is no False in the vector
        return True

    # Check if all values after the first False are also False
    return not vec[first_false_index:].any() and vec[:first_false_index].all()

def time_features(
    dates: pd.Series,
    main_df=None,
    use_features=["year", "month", "day", "weekday", "hour", "minute"],
    time_col_name="Datetime",
):

    if main_df is None:
        main_df = pd.DataFrame({})
    else:
        main_df = main_df.copy()
    years = dates.apply(lambda row: row.year)
    max_year = years.max()
    min_year = years.min()
    
    norm_f = lambda r: 2.0*r - 1.0
    norm_f = lambda r: r

    if "year" in use_features:
        main_df["year"] = dates.apply(
            lambda row: (row.year - min_year) / max(1.0, (max_year - min_year))
        )

    if "month" in use_features:
        main_df["month"] = dates.apply(
           lambda row: norm_f((row.month - 1) / 11.0), 1
        )
    if "day" in use_features:
        main_df["day"] = dates.apply(
          lambda row: norm_f((row.day - 1) / 30.0), 1
        )
        
    if "weekday" in use_features:
        main_df["weekday"] = dates.apply(
            lambda row: norm_f(row.weekday() / 6.0), 1
        )
    if "hour" in use_features:
        main_df["hour"] = dates.apply(
          lambda row: norm_f((row.hour) / 23.0), 1
        )
        
    if "minute" in use_features:
        main_df["minute"] = dates.apply(
            lambda row: ((row.minute) / 59.0), 1
        )

    main_df[time_col_name] = dates
    return main_df

In [3]:
class IndexSequencer:
    
    def __init__(
        self, F: torch.Tensor,  
        ts_in: int, 
        ts_out: int,
        dt_ts_out: List[int] = None,
        step: int = 1) -> None:
        
        self.F = F

        if dt_ts_out is not None:
            dt_ts_out.sort()
            dt_ts_out = np.array(dt_ts_out)
            dt_max = dt_ts_out[-1]
            upper_limit = max(ts_out, dt_max + 1)
        else:
            upper_limit = ts_out
            
        N_samples = F.shape[0]
        
        i = torch.arange(0, N_samples - (ts_in + upper_limit) + 1, step=step)
        self.start_end_index = torch.stack ([i, i + (ts_in + upper_limit)], dim=1)
        
        self.N_sequences = len(self.start_end_index)
        
    def __len__(self):
        return self.N_sequences
      
      
    def sequences_batch(self, index_subset: torch.Tensor | np.ndarray) -> Tuple[torch.Tensor]:
        sel = self.start_end_index[index_subset]
        return self._create_sequences(sel)
    
    @abstractmethod
    def _create_sequences(self, sel: List[Tuple[int]]) -> Tuple[torch.Tensor]:
        pass

   
class AutoregressiveSequencer(IndexSequencer):
    
    def __init__(
        self, F: torch.Tensor,
        ts_in: int, 
        ts_out: int,
        T: torch.Tensor=None,
        in_features_mask=None, 
        out_features_mask=None,
        step: int=10) -> None:
        
        super().__init__(F=F, ts_in=ts_in, ts_out=ts_out, step=step)
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.in_features_mask = in_features_mask
        self.out_features_mask = out_features_mask
        self.T = F
    
    def _create_sequences(self, sel: List[Tuple[int]]) -> Tuple[torch.Tensor]:
        ts_in, ts_out = self.ts_in, self.ts_out
        F, T = self.F, self.T
        
        f_unfolded, t_unfolded = [], []
        for i, j in sel:
            f_unfolded.append(F[i : i + ts_in, ...])
            t_unfolded.append(T[i + ts_in : j, ...])
        
        f_unfolded = torch.stack(f_unfolded, dim=0)
        t_unfolded = torch.stack(t_unfolded, dim=0)
        
        if self.in_features_mask is not None:
            f_unfolded = f_unfolded[..., self.in_features_mask]
            
        if self.out_features_mask is not None:
            t_unfolded = t_unfolded[..., self.out_features_mask]
        
        return f_unfolded, t_unfolded

class ContextBatchDataset(torch.utils.data.Dataset):
    def __init__(self, 
        Features: torch.Tensor, 
        Context: torch.Tensor,
        ts_in: int,
        ts_out:int,
        out_features_mask: Optional[torch.Tensor] = None,
        step: int = 1,
        raw_data: Optional[pd.DataFrame] = None):
        
        self.raw_data = raw_data
        self.F_seq = AutoregressiveSequencer(Features, ts_in, ts_out, step=step, out_features_mask=out_features_mask)
        self.C_seq = AutoregressiveSequencer(Context, ts_in, ts_out, step=step)
        
        self.ts_in = ts_in
        
    def get_raw_data(self, batch_idx: List[int]):
        if self.raw_data is not None:
            
            ts_in = self.ts_in
            sel = self.F_seq.start_end_index[batch_idx]
            
            f_unfolded, t_unfolded = [], []
            for i, j in sel:
                i, j  = i.item(), j.item()
                f_unfolded.append(self.raw_data.iloc[i : i + ts_in, ...])
                t_unfolded.append(self.raw_data.iloc[i + ts_in : j, ...])
            
            return f_unfolded, t_unfolded
        else:
            raise ValueError("No raw data available")
    
    def __len__(self):
        return len(self.F_seq)

    def __getitem__(self, batch_idx: List[int]): 
        F_f, F_t = self.F_seq.sequences_batch(batch_idx)
        C_f, C_t = self.C_seq.sequences_batch(batch_idx)
        return C_f, F_f, C_t, F_t


class BuildingEnergyDataModule(pl.LightningDataModule):
    def __init__(self,
                 data_dir: str,
                 test_ratio: float = 0.5,
                 val_ratio: float = 0.084,
                 train_ratio: float = 1.0,
                 train_scaled: Optional[None] = None,
                 val_scaled: Optional[None] = None,
                 test_scaled: Optional[None] = None,
                 batch_size=64,
                 samples_per_day=None,
                 trend_decomp_rolling_window=384,
                 ts_in=336,
                 ts_out=48,
                 step_train=1,
                 step_val=48,
                 step_test=48,
                 shuffle=True,
                 drop_last=False,
                 num_workers=0,
                 data_split_strategy="default",
                 include_bl_forecast=False,
                 datetime_features=True,
                 forecast_feature_name="Site Total Energy (kWh)",
                 forecast_all_features=False,
                 trend_decomp="disabled",
                 subsample_strategy=None,
                 spatial_aggregation: Optional[str] = None):

        super().__init__()
        self.test_ratio = test_ratio
        self.val_ratio = val_ratio
        self.train_ratio = train_ratio

        if train_ratio < 1.0:
            assert data_split_strategy == "scale_training_set", "Only scale_training_set strategy is supported"

        self.samples_per_day = samples_per_day
        self.trend_decomp_rolling_window = trend_decomp_rolling_window
        self.subsample_strategy = subsample_strategy

        self.train_scaled = train_scaled
        self.val_scaled = val_scaled
        self.test_scaled = test_scaled
        self.batch_size = batch_size
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.step_train = step_train
        self.step_val = step_val
        self.step_test = step_test
        self.forecast_feature_name = forecast_feature_name
        self.forecast_all_features = forecast_all_features
        self.data_split_strategy = data_split_strategy

        self.shuffle = shuffle
        self.drop_last = drop_last

        self.datetime_features = datetime_features

        self.num_workers = num_workers
        self.include_bl_forecast = include_bl_forecast
        self.trend_decomp = trend_decomp
        self.data_dir = data_dir
        self.spatial_aggregation = spatial_aggregation

    def load_raw_data(self, data_dir):
        data_path = data_dir
        data = pd.read_csv(data_path, sep=",")

        data['Unnamed: 0'] = pd.to_datetime(data['Unnamed: 0'])
        data.set_index('Unnamed: 0', inplace=True)

        return data

    def setup(self, stage: Optional[str] = None):

        if self.train_scaled is None:
            data = self.load_raw_data(self.data_dir)

            # use this to downsample the data
            data_samples_per_day = data.index.to_series().groupby(data.index.date).count().iloc[0]
            samples_per_day = data_samples_per_day

            assert self.samples_per_day is None, "Subsampling not implemented"
            assert self.subsample_strategy is None, "Not implemented"

            if not (self.subsample_strategy == "mean_1h" or self.subsample_strategy == None):
                raise ValueError("Only mean_1h is supported, make sure the dataset is aligned with the small one")

            if self.subsample_strategy == "select_1h":
                hourly = pd.Series(data=pd.date_range(start=data.index.min(), end=data.index.max(), freq='1h'))
                data = data.loc[hourly]
            if self.subsample_strategy == "mean_1h":
                data['Bahnlast'] = data['Bahnlast'].shift(-2).ffill()
                data = data.shift(1).bfill().resample("1h").mean()
                data = data.resample('1H').mean()

            self.data = {"full": data}

            """
            WARNING: comumns need to be in order:
            Bahnlast <forecast targets>, covaraites <contexts> ...
            The unintuitive part is that if there are N variables in the context
            sequence (the encoder input) and K (K < N) variables in the target sequence
            (the decoder input), then this implementation of Local Cross Attn
            assumes that the first K variables in the context correspond to the
            first K in the target. This means that if the context sequence is shape 
            (batch, length, N), then context[:, :, :K] gets you the context of the
            K target variables (target[..., i] is the same variable
            as context[..., i]). If this isn't true the model will still train but
            you will be connecting variables by cross attention in a very arbitrary
            way. Note that the built-in CSVDataset *does* account for this and always
            puts the target variables in the same order in the lowest indices of both
            sequences. ** If your target variables do not appear in the context sequence
            Local Cross Attention should almost definitely be turned off
            (--local_cross_attn none) **.
            """

            if self.spatial_aggregation is not None:
                print("Spatial aggregation")

            if self.trend_decomp == "disabled":
                gt = data[[
                    'Site Total Energy (kWh)',
                ]]
                self.data = {"full": data, "gt": gt.copy()}
            else:
                raise ValueError("Unknown trend decomposition")

            if not self.forecast_all_features:
                # self.features_mask = (data.columns == "Bahnlast")
                self.bahnlast_mask = (data.columns == self.forecast_feature_name)
                self.bahnlast_mask_orig = (data.columns == self.forecast_feature_name)
            else:
                self.bahnlast_mask = np.ones(data.shape[1], dtype=bool)
                self.bahnlast_mask_orig = np.ones(data.shape[1], dtype=bool)

            assert check_boolean_vector(self.bahnlast_mask), "Should be boolean vector in order 1111...000"
            assert check_boolean_vector(self.bahnlast_mask_orig), "Should be boolean vector in order 1111...000"

            test_ratio = self.test_ratio
            val_ratio = self.val_ratio
            train_ratio = self.train_ratio

            cut_to_days = lambda x: (x // samples_per_day) * samples_per_day
            N_samples = cut_to_days(len(data))
            if self.data_split_strategy == "default":
                # if you modify this: this is very fragile to keep the DS day aligned!
                N_test = cut_to_days(round(test_ratio * N_samples))
                N_val = cut_to_days(round(val_ratio * N_samples))
                N_train = cut_to_days(N_samples - N_test - N_val)
                print(N_train)

                r_train = range(0, N_train)
                r_val = range(N_train, N_train + N_val)
                r_test = range(N_train + N_val, N_train + N_val + N_test)

            print("--" * 40)
            print("Dataset Split Summary:")
            print(f"Train: {len(r_train)} , Start Date: {data.index[r_train[0]]} , End Date: {data.index[r_train[-1]]}")
            print(f"Val: {len(r_val)} , Start Date: {data.index[r_val[0]]} , End Date: {data.index[r_val[-1]]}")
            print(f"Test: {len(r_test)} , Start Date: {data.index[r_test[0]]} , End Date: {data.index[r_test[-1]]}")
            print("--" * 40)

            self.r_train, self.r_val, self.r_test = r_train, r_val, r_test
            self.train = data.iloc[r_train, :]
            self.val = data.iloc[r_val, :]
            self.test = data.iloc[r_test, :]

            scaler = MinMaxScaler()
            scaler.fit(self.train)
            self.train_scaled = scaler.transform(self.train)
            self.val_scaled = scaler.transform(self.val)
            self.test_scaled = scaler.transform(self.test)

            if self.datetime_features:
                dtf = time_features(
                    pd.Series(data.index),
                    use_features=["month", "day", "weekday", "hour"])
                self.dtf_train = dtf.iloc[r_train, :]
                self.dtf_val = dtf.iloc[r_val, :]
                self.dtf_test = dtf.iloc[r_test, :]

                self.scaler_mask = np.concatenate(
                    (np.ones(self.train_scaled.shape[1], dtype=bool), np.zeros(4, dtype=bool)))

                self.bahnlast_mask = np.concatenate(
                    (self.bahnlast_mask, np.zeros(4, dtype=bool)))

                self.train_scaled = np.concatenate(
                    (self.train_scaled, self.dtf_train[['hour', 'day', 'weekday', 'month']].values), axis=1)
                self.val_scaled = np.concatenate(
                    (self.val_scaled, self.dtf_val[['hour', 'day', 'weekday', 'month']].values), axis=1)
                self.test_scaled = np.concatenate(
                    (self.test_scaled, self.dtf_test[['hour', 'day', 'weekday', 'month']].values), axis=1)

            self.scaler = scaler

            def to_ds(ds, step, raw_data):

                # out_features_mask = torch.from_numpy(data.columns == self.forecast_feature_name).to(torch.bool) \
                #     if self.forecast_all_features else None

                out_features_mask = None

                ds = ContextBatchDataset(
                    Features=torch.from_numpy(ds[:, self.bahnlast_mask]).to(torch.float32),
                    Context=torch.from_numpy(ds[:, ~self.bahnlast_mask]).to(torch.float32),
                    out_features_mask=out_features_mask,
                    ts_in=self.ts_in,
                    ts_out=self.ts_out,
                    step=step,
                    raw_data=raw_data
                )
                return ds

            self.datasets = dict(
                train_test=to_ds(self.train_scaled, self.step_test, self.train),
                train=to_ds(self.train_scaled, self.step_train, self.train),
                val=to_ds(self.val_scaled, self.step_val, self.val),
                test=to_ds(self.test_scaled, self.step_test, self.test),
            )

    def inverse_scale_bahnlast(self, X):
        x = np.zeros((X.shape[0], self.train_scaled.shape[1]))
        x[:, self.bahnlast_mask] = X

        if self.datetime_features:
            res = self.scaler.inverse_transform(x[:, self.scaler_mask])
        else:
            res = self.scaler.inverse_transform(x)

        return res[:, self.bahnlast_mask_orig]

    def _create_dataloader(self, ds, batch_size, shuffle=True):

        sampler = BatchSampler(
            RandomSampler(ds) if shuffle else SequentialSampler(ds),
            batch_size=batch_size,
            drop_last=self.drop_last
        )
        loader_args = dict(
            sampler=sampler,
            num_workers=self.num_workers,
            # persistent_workers=True,
            batch_size=None  # otherwise it adds a dimension with size 1 x ...
            # due to the batch sampler providing the batch dimension already
        )
        return DataLoader(ds, **loader_args)

    def train_dataloader(self):
        return self._create_dataloader(self.datasets['train'], batch_size=self.batch_size)

    def val_dataloader(self):
        return self._create_dataloader(self.datasets['val'], batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return self._create_dataloader(self.datasets['test'], batch_size=self.batch_size, shuffle=False)

    def test_small_dataloader(self):
        return self._create_dataloader(self.datasets['test_small'], batch_size=self.batch_size, shuffle=False)

In [4]:
class SpatialDataModule(pl.LightningDataModule):
    def __init__(self, 
        test_ratio: float = 0.2,
        val_ratio: float = 0.07,
        train_ratio: float = 1.0,
        train_scaled: Optional[None] = None, 
        val_scaled: Optional[None] = None, 
        test_scaled: Optional[None] = None, 
        batch_size=64, 
        samples_per_day = 24,
        trend_decomp_rolling_window = 384,
        ts_in=24*7, 
        ts_out=24,
        step_train=1,
        step_val=24,
        step_test=24,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        data_split_strategy="scale_training_set",
        include_bl_forecast=False,
        datetime_features=True,
        forecast_feature_name="Bahnlast",
        forecast_all_features=False,
        trend_decomp="disabled",
        subsample_strategy="mean_1h",
        spatial_aggregation: Optional[str] = None):
        
        super().__init__()
        self.test_ratio = test_ratio
        self.val_ratio = val_ratio
        self.train_ratio = train_ratio
        
        self.samples_per_day = samples_per_day
        self.trend_decomp_rolling_window = trend_decomp_rolling_window
        self.subsample_strategy = subsample_strategy
        
        self.train_scaled = train_scaled
        self.val_scaled = val_scaled
        self.test_scaled = test_scaled
        self.batch_size = batch_size
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.step_train = step_train
        self.step_val = step_val
        self.step_test = step_test
        self.forecast_feature_name = forecast_feature_name
        self.forecast_all_features = forecast_all_features
        self.data_split_strategy = data_split_strategy
        
        self.shuffle = shuffle
        self.drop_last = drop_last
        
        self.datetime_features = datetime_features
        
        self.num_workers = num_workers
        self.include_bl_forecast = include_bl_forecast
        self.trend_decomp = trend_decomp
        self.spatial_aggregation = spatial_aggregation

    def setup(self, stage: Optional[str] = None):

        if self.train_scaled is None:
            data = pd.read_csv("datasets/railway.csv", sep=",")
            data.datetime = pd.to_datetime(data.datetime)
            data.set_index("datetime", inplace=True)
            self.data = {"full": data}
            
            samples_per_day = self.samples_per_day
            
            if self.trend_decomp == "disabled":
                bl = data['Bahnlast']
                data.rename(columns={"Bahnlast": "Bahnlast_oldpos"}, inplace=True)
                data.insert(loc=0, column='Bahnlast', value=bl)
                data.drop(columns=['Bahnlast_oldpos'], inplace=True)
                self.data = {"full": data, "gt": bl.copy()}
                
                
            elif self.trend_decomp == "mean":
                S = self.trend_decomp_rolling_window
                self.prev_mean = data['Bahnlast'].rolling(S).mean().bfill()
                self.prev_mean_shifted = data['Bahnlast'].rolling(S).mean().shift(samples_per_day).bfill()
                
                decomp = data['Bahnlast'] - self.prev_mean_shifted
                data.insert(loc=0, column='Bahnlast_decomp', value=decomp)
                
                self.data = {"full": data, "gt": data['Bahnlast'].copy()}
                data.drop(columns=['Bahnlast'], inplace=True)
            else:
                raise ValueError("Unknown trend decomposition")
                
            if not self.forecast_all_features:
                self.bahnlast_mask = (data.columns == self.forecast_feature_name)
                self.bahnlast_mask_orig = (data.columns == self.forecast_feature_name)
            else:
                self.bahnlast_mask = np.ones(data.shape[1], dtype=bool)
                self.bahnlast_mask_orig = np.ones(data.shape[1], dtype=bool)

            test_ratio = self.test_ratio
            val_ratio = self.val_ratio
            train_ratio = self.train_ratio
            
            cut_to_days = lambda x: (x // samples_per_day) * samples_per_day
            N_samples = cut_to_days(len(data))
            if self.data_split_strategy == "default":
                N_test = cut_to_days(round(test_ratio * N_samples))
                N_val = cut_to_days(round(val_ratio * N_samples))
                N_train = cut_to_days(N_samples - N_test - N_val)
                
                r_train = range(0,N_train)
                r_val = range(N_train,N_train+N_val)
                r_test = range(N_train+N_val,N_train+N_val+N_test)
                r_test_small = range(data.index.get_loc("2023-01-24 00:00:00"), data.index.get_loc("2023-07-31 00:00:00"))
                

            elif self.data_split_strategy == "scale_training_set":
                ################################################
                #    Train  |  Val  |  Test_small  | Test
                #       xxxx|xxxxxxx|xxxxxxxxxxxxxxxxxxxxx
                #   xxxxxxxx|xxxxxxx|xxxxxxxxxxxxxxxxxxxxx train 
                #
                ################################################
                
                # we keep a relatively long test set
                r_test_small = range(data.index.get_loc("2023-01-24 00:00:00"), data.index.get_loc("2023-07-31 00:00:00"))
                r_test = range(data.index.get_loc("2023-01-24 00:00:00"), N_samples)
                
                # we cut a small validation set
                N_val = cut_to_days(round(val_ratio * (N_samples - len(r_test))))
                Len_train = N_samples - N_val - len(r_test)
                N_train = cut_to_days(round(train_ratio * (Len_train)))
                   
                r_train = range(Len_train-N_train,Len_train)
                r_val = range(Len_train,Len_train+N_val)
                
            elif self.data_split_strategy == "covid_training_set":
                # we keep a relatively long test set
                r_test_small = range(data.index.get_loc("2023-01-24 00:00:00"), data.index.get_loc("2023-07-31 00:00:00"))
                r_test = range(data.index.get_loc("2023-01-24 00:00:00"), N_samples)
                
                # we cut a small validation set
                N_val = cut_to_days(round(val_ratio * (N_samples - len(r_test))))
                Len_train = N_samples - N_val - len(r_test)
                   
                r_train = range(data.index.get_loc("2019-12-01 00:00:00"), data.index.get_loc("2021-07-31 00:00:00"))
                r_val = range(Len_train,Len_train+N_val)

            self.r_train, self.r_val, self.r_test, self.r_test_small = r_train, r_val, r_test, r_test_small 
            self.train = data.iloc[r_train, :]
            self.val = data.iloc[r_val, :]
            self.test = data.iloc[r_test, :]
            self.test_small = data.iloc[r_test_small, :]

            
                            
            scaler = MinMaxScaler()
            scaler.fit(self.train)
            self.train_scaled = scaler.transform(self.train)
            self.val_scaled = scaler.transform(self.val)
            self.test_scaled = scaler.transform(self.test)
            self.test_small_scaled = scaler.transform(self.test_small)
            
            if self.datetime_features:
                dtf = time_features(
                    pd.Series(data.index),
                    use_features=["month", "day", "weekday", "hour"])
                self.dtf_train = dtf.iloc[r_train, :]
                self.dtf_val = dtf.iloc[r_val, :]
                self.dtf_test = dtf.iloc[r_test, :]
                self.dtf_test_small = dtf.iloc[r_test_small, :]
                
                self.scaler_mask = np.concatenate(
                    (np.ones(self.train_scaled.shape[1], dtype=bool), np.zeros(4, dtype=bool)))
                
                self.bahnlast_mask = np.concatenate(
                    (self.bahnlast_mask, np.zeros(4, dtype=bool)))
                
                self.train_scaled = np.concatenate(
                    (self.train_scaled, self.dtf_train[['hour', 'day', 'weekday', 'month']].values), axis=1)
                self.val_scaled = np.concatenate(
                    (self.val_scaled, self.dtf_val[['hour', 'day', 'weekday', 'month']].values), axis=1)
                self.test_scaled = np.concatenate(
                    (self.test_scaled, self.dtf_test[['hour', 'day', 'weekday', 'month']].values), axis=1)
                self.test_small_scaled = np.concatenate(
                    (self.test_small_scaled, self.dtf_test_small[['hour', 'day', 'weekday', 'month']].values), axis=1)
            
            self.scaler = scaler
            
            def to_ds(ds, step, raw_data):
                
                out_features_mask = None
                
                ds = ContextBatchDataset(
                    Features=torch.from_numpy(ds[:,self.bahnlast_mask]).to(torch.float32),
                    Context=torch.from_numpy(ds[:,~self.bahnlast_mask]).to(torch.float32),
                    out_features_mask=out_features_mask,
                    ts_in=self.ts_in,
                    ts_out=self.ts_out,
                    step=step,
                    raw_data=raw_data
                )
                return ds

        
            self.datasets = dict(
                train_test = to_ds(self.train_scaled, self.step_test, self.train),
                train = to_ds(self.train_scaled, self.step_train, self.train),
                val = to_ds(self.val_scaled, self.step_val, self.val),
                test = to_ds(self.test_scaled, self.step_test, self.test),
                test_small = to_ds(self.test_small_scaled, self.step_test, self.test)
            )

            
            
    def inverse_scale_bahnlast(self, X):
        x = np.zeros((X.shape[0], self.train_scaled.shape[1]))
        x[:, self.bahnlast_mask] = X
        
        if self.datetime_features:
            res = self.scaler.inverse_transform(x[:, self.scaler_mask])
        else:
            res = self.scaler.inverse_transform(x)
                   
        return res[:,self.bahnlast_mask_orig]
    
        
    def _create_dataloader(self, ds, batch_size, shuffle=True):
        
        sampler =  BatchSampler(
            RandomSampler(ds) if shuffle else SequentialSampler(ds),
            batch_size=batch_size, 
            drop_last=self.drop_last
        ) 
        loader_args = dict(             
            sampler=sampler,
            num_workers=self.num_workers,
            batch_size=None # otherwise it adds a dimension with size 1 x ...
        ) 
        return DataLoader(ds,  **loader_args)


    def train_dataloader(self):
        return self._create_dataloader(self.datasets['train'], batch_size=self.batch_size)

    def val_dataloader(self):
        return self._create_dataloader(self.datasets['val'], batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return self._create_dataloader(self.datasets['test'], batch_size=self.batch_size, shuffle=False)

    def test_small_dataloader(self):
        return self._create_dataloader(self.datasets['test_small'], batch_size=self.batch_size, shuffle=False)


# Regression Benchmark Railway

In [5]:
dm = SpatialDataModule(ts_in=24)
dm.setup()

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/railway.csv'

In [8]:
def dl_to_sklearn_with_context(dl):
    X, y = [], []
    for batch in iter(dl): 
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([torch.concat([y_c, x_c], dim=2).flatten(1), x_t.flatten(1)], dim=1))
        y.append(y_t.flatten(1))
    
    return torch.concat(X).numpy(), torch.concat(y).numpy()


def dl_to_sklearn__with_context_for_tabpfn(dl, lag=0):
    X, y = [], []
    for batch in iter(dl): 
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([x_t[:, lag, :], y_c.squeeze(2)], dim=1))
        y.append(y_t[:,lag,:])
        
    
    return torch.concat(X).numpy(), torch.concat(y).numpy()


def dl_to_sklearn_no_context(dl):
    X, y = [], []
    for batch in iter(dl): 
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([y_c, x_c], dim=2).flatten(1))
        y.append(y_t.flatten(1))
    
    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [9]:
batch = next(iter(dm.train_dataloader()))
x_c, y_c, x_t, y_t = batch

In [10]:
x_t.shape, y_t.shape, x_c.shape, y_c.shape

(torch.Size([64, 24, 56]),
 torch.Size([64, 24, 1]),
 torch.Size([64, 24, 56]),
 torch.Size([64, 24, 1]))

In [11]:
torch.concat([x_t[:, 0, :], y_c.squeeze(2)], dim=1).shape, y_c.squeeze(2)[:,0].shape

(torch.Size([64, 80]), torch.Size([64]))

In [12]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from tqdm import tqdm

# Assuming there is a TabPFNRegressor (if not, a different regressor should be used)
from tabpfn import TabPFNRegressor  

feature_mode = "exogen"  # "endogen" or "exogen"
model_name = "tabpfn"
seed = -1
results = []
N_samples = 1000
factor_agg = pd.read_csv("datasets/railway_agg.csv")['Bahnlast'].abs().mean()
factor_full = pd.read_csv("datasets/railway.csv")['Bahnlast'].abs().mean()
# factor_full = 1



for lag in tqdm(range(24)):
    
    
    if feature_mode == "endogen":
        X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
        X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
    elif feature_mode == "exogen":
        X_train, y_train = dl_to_sklearn__with_context_for_tabpfn(dm.train_dataloader(), lag=lag)
        X_test, y_test = dl_to_sklearn__with_context_for_tabpfn(dm.test_dataloader(), lag=lag)
    else:
        raise ValueError("Feature mode unknown")
    
    if N_samples is not None:
        X_train, y_train = X_train[-N_samples:], y_train[-N_samples:]
        X_test, y_test = X_test[:N_samples], y_test[:N_samples]
    
    # Initialize the regressor
    regressor = TabPFNRegressor(ignore_pretraining_limits=True)  
    regressor.fit(X_train, y_train)

    # Predict on the test set
    y_test_hat = regressor.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_test_hat)
    r2 = r2_score(y_test, y_test_hat)

    print("Mean Squared Error (MSE):", mse)
    print("R² Score:", r2)



    y_test_hat_rs = dm.inverse_scale_bahnlast(rearrange(y_test_hat, "B -> (B) 1"))
    y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B F -> (B F) 1"))



    results.append({"exp_name": f"{model_name}_{feature_mode}",
                    "seed": seed,
                    "lag" : lag,
                    "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean() *100 / factor_full, 
                    "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs)**2).mean() *100 / factor_full, })

df = pd.DataFrame(results).set_index(["exp_name", "seed"])
df
# df.to_csv(f"railway/{model_name}_{feature_mode}.csv")

  0%|          | 0/24 [00:00<?, ?it/s]c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs 

Mean Squared Error (MSE): 0.00041830487
R² Score: 0.9187125353428957


c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_produ

Mean Squared Error (MSE): 0.00039220415
R² Score: 0.913246422488809


c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_produ

...

c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_produ

Mean Squared Error (MSE): 0.0010470862
R² Score: 0.7971586455301284


c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Raffael\miniforge3\envs\torch2_cuda12.1\lib\site-packages\tabpfn\model\multi_head_attention.py:710: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:252.)
  attention_head_outputs = torch.nn.functional.scaled_dot_produ

Mean Squared Error (MSE): 0.0006817422
R² Score: 0.8631242376224668


lag  normalized/test/mae  normalized/test/mse
exp_name      seed                                               
tabpfn_exogen -1      0             2.870399            32.856102
              -1      1             2.706923            30.806001
              -1      2             2.729646            33.934442
              -1      3             3.767638            58.013597
              -1      4             3.148687            43.922712
              -1      5             3.648530            59.179882
              -1      6             4.628617            87.702621
              -1      7             3.356143            52.591161
              -1      8             3.441536            49.104260
              -1      9             4.104033            68.866999
              -1     10             4.277987            77.639181
              -1     11             4.004156            67.078633
              -1     12             3.907228            63.643269
              -1     13             3.659900            56.709394
              -1     14             4.023418            68.292618
              -1     15             4.227072            75.932711
              -1     16             4.488769            81.845974
              -1     17             3.533414            55.220625
              -1     18             4.314496            77.244198
              -1     19             3.927556            71.542811
              -1     20             4.047814            66.476304
              -1     21             3.343478            48.481074
              -1     22             4.562335            82.244251
              -1     23             3.414897            53.548002

In [13]:
df.mean()

lag                    11.500000
normalized/test/mae     3.755611
normalized/test/mse    60.953201
dtype: float64

In [1]:
df

NameError: name 'df' is not defined

In [ ]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import os
import random
from joblib import dump

seeds = [72, 88, 101, 666, 226688]

for model_name in ["LinearRegression", "KNN", "MLP"]:
    for feature_mode in ["endogen", "exogen"]:
        results = []
        for seed in seeds:
            random.seed(seed)
            os.environ['PYTHONHASHSEED'] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        
            if feature_mode == "endogen":
                X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
            elif feature_mode == "exogen":
                X_train, y_train = dl_to_sklearn_with_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_with_context(dm.test_dataloader())
            else:
                raise ValuerError("Feature mode unknown")
        
            if model_name == "Catboost":
                model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=4, silent=True, loss_function='MultiRMSE', random_seed=seed)
            elif model_name == "XGBoost":
                model = XGBRegressor(verbosity=0, objective='reg:squarederror', n_estimators=1000, device="cuda", random_state=seed)
            elif model_name == "LinearRegression":
                model = LinearRegression()
            elif model_name == "KNN":
                model = KNeighborsRegressor()
            elif model_name == "MLP":
                model = MLPRegressor(hidden_layer_sizes=(100,100), batch_size=64, max_iter=1000, early_stopping=True, validation_fraction=0.2, random_state=seed)
            elif model_name = "TabPFN":
                model = TabPFNRegressor()
            else:
                raise ValuerError("Model name unknown")
        
            model.fit(X_train, y_train)
        
            print("Evaluating model with seed", seed)
        
            y_test_hat = model.predict(X_test)

            np.save(f"railway/{model_name}_{feature_mode}_{seed}_forecasts.npy", y_test_hat)
            if model_name == "Catboost":
                model.save_model(f"railway/Catboost_{feature_mode}_{seed}_model.json")
            elif model_name == "XGBoost":
                model.save_model(f"railway/XGBoost_{feature_mode}_{seed}_model.json")
            elif model_name == "LinearRegression":
                dump(model, f"railway/LinearRegression_{feature_mode}_{seed}_model.joblib") 
            elif model_name == "KNN":
                dump(model, f"railway/KNN_{feature_mode}_{seed}_model.joblib")
            elif model_name == "MLP":
                dump(model, f"railway/MLP_{feature_mode}_{seed}_model.joblib")
            else:
                raise ValuerError("Model name unknown")

            y_test_hat_rs = dm.inverse_scale_bahnlast(rearrange(y_test_hat, "B T -> (B T) 1"))
            y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B T -> (B T) 1"))
        
            results.append({"exp_name": f"{model_name}_{feature_mode}",
                            "seed": seed,
                            "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(), 
                            "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs)**2).mean()})
        
        df = pd.DataFrame(results).set_index(["exp_name", "seed"])
        df.to_csv(f"railway/{model_name}_{feature_mode}.csv")

### Persistence

In [ ]:
# PERSISTENCE
forecast_horizon = 24
seasonal_lag = 168
stride = 24

# Generate test data
dm = SpatialDataModule()
dm.setup()

_, _ = dl_to_sklearn_no_context(dm.train_dataloader())
_, y_test = dl_to_sklearn_no_context(dm.test_dataloader())

# Load target data
target_data = pd.read_csv("C:/Users/vole/Downloads/Building_Operation_Data/railway.csv").set_index("datetime")['Bahnlast']
target_data.index = pd.to_datetime(target_data.index)

# Vectorized extraction of the lagged forecast matrix
persistence_base = target_data.shift(seasonal_lag)
persistence_matrix = pd.concat([persistence_base.shift(-i) for i in range(forecast_horizon)], axis=1).dropna()

# Rearrange
y_test_hat_rs = rearrange(persistence_matrix[::stride].loc["2023-01-24 00:00:00":][int(seasonal_lag/forecast_horizon):].values, "B T -> (B T) 1")
y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B T -> (B T) 1"))

for feature_mode in ["endogen", "exogen"]:
    benchmark = []
    for seed in seeds:
        np.save(f"railway/WeeklyNaive_{feature_mode}_{seed}_forecasts.npy", persistence_matrix[::stride].loc["2023-01-24 00:00:00":][int(seasonal_lag/forecast_horizon):].values)
        benchmark.append({"exp_name": f"WeeklyNaive_{feature_mode}",
                            "seed": seed,
                            "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(), 
                            "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs)**2).mean()})
    df = pd.DataFrame(benchmark).set_index(["exp_name", "seed"])
    df.to_csv(f"railway/WeeklyNaive_{feature_mode}.csv")

# Regression Benchmarks Buildings

In [8]:
dm = BuildingEnergyDataModule(data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv", test_ratio=0.5)
dm.load_raw_data(data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv")
dm.setup()

14592
--------------------------------------------------------------------------------
Dataset Split Summary:
Train: 14592 , Start Date: 2002-01-01 00:00:00 , End Date: 2002-10-31 23:30:00
Val: 2928 , Start Date: 2002-11-01 00:00:00 , End Date: 2002-12-31 23:30:00
Test: 17520 , Start Date: 2003-01-01 00:00:00 , End Date: 2003-12-31 23:30:00
--------------------------------------------------------------------------------


C:\Users\vole\AppData\Local\Temp\ipykernel_11876\2783055339.py:41: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  main_df["month"] = dates.apply(
C:\Users\vole\AppData\Local\Temp\ipykernel_11876\2783055339.py:45: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  main_df["day"] = dates.apply(
C:\Users\vole\AppData\Local\Temp\ipykernel_11876\2783055339.py:50: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  main_df["weekday"] = dates.apply(
C:\Users\vole\AppData\Local\Temp\ipykernel_11876\2783055339.py:54: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version

In [9]:
def dl_to_sklearn_with_context(dl):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([torch.concat([y_c, x_c], dim=2).flatten(1), x_t.flatten(1)], dim=1))
        y.append(y_t.flatten(1))
    
    return torch.concat(X).numpy(), torch.concat(y).numpy()

def dl_to_sklearn_no_context(dl):
    X, y = [], []
    for i, batch in enumerate(iter(dl)): 
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([y_c, x_c], dim=2).flatten(1))
        y.append(y_t.flatten(1))
    
    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [12]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
import lightgbm as lgb
import os
import random
from joblib import dump

seeds = [72, 88, 101, 666, 226688]

for model_name in ["LinearRegression", "KNN", "MLP", "XGBoost", "Catboost"]:
    for feature_mode in ["endogen", "exogen"]:
        # Train and evaluate models with different random seeds
        results = []
        for seed in seeds:
            random.seed(seed)
            os.environ['PYTHONHASHSEED'] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        
            if feature_mode == "endogen":
                X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
            elif feature_mode == "exogen":
                X_train, y_train = dl_to_sklearn_with_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_with_context(dm.test_dataloader())
            else:
                raise ValuerError("Feature mode unknown")
        
            if model_name == "Catboost":
                model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=4, silent=True, loss_function='MultiRMSE', random_seed=seed)
            elif model_name == "XGBoost":
                model = XGBRegressor(verbosity=0, objective='reg:squarederror', n_estimators=1000, device="cuda", random_state=seed)
            elif model_name == "LightGBM":
                model = model = MultiOutputRegressor(lgb.LGBMRegressor(boosting_type='gbdt', n_estimators=1000, verbose=-1))
            elif model_name == "LinearRegression":
                model = LinearRegression()
            elif model_name == "KNN":
                model = KNeighborsRegressor()
            elif model_name == "MLP":
                model = MLPRegressor(hidden_layer_sizes=(100,100), batch_size=64, max_iter=1000, early_stopping=True, validation_fraction=0.2, random_state=seed)
            else:
                raise ValuerError("Model name unknown")
        
            model.fit(X_train, y_train)
        
            print("Evaluating model with seed", seed)
        
            y_test_hat = model.predict(X_test)

            np.save(f"building/{model_name}_{feature_mode}_{seed}_forecasts.npy", y_test_hat)
            if model_name == "Catboost":
                model.save_model(f"building/Catboost_{feature_mode}_{seed}_model.json")
            elif model_name == "XGBoost":
                model.save_model(f"building/XGBoost_{feature_mode}_{seed}_model.json")
            elif model_name == "LightGBM":
                dump(model, f"building/LightGBM_{feature_mode}_{seed}_model.joblib")
            elif model_name == "LinearRegression":
                dump(model, f"building/LinearRegression_{feature_mode}_{seed}_model.joblib") 
            elif model_name == "KNN":
                dump(model, f"building/KNN_{feature_mode}_{seed}_model.joblib")
            elif model_name == "MLP":
                dump(model, f"building/MLP_{feature_mode}_{seed}_model.joblib")
            else:
                raise ValuerError("Model name unknown")

            y_test_hat_rs = rearrange(y_test_hat, "B T -> (B T) 1")
            y_test_rs = rearrange(y_test, "B T -> (B T) 1")
        
            results.append({"exp_name": f"{model_name}_{feature_mode}",
                            "seed": seed,
                            "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(), 
                            "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs)**2).mean()})
        
        df = pd.DataFrame(results).set_index(["exp_name", "seed"])
        df.to_csv(f"building/{model_name}_{feature_mode}.csv")

C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

Evaluating model with seed 72


C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

Evaluating model with seed 88


C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

...

C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

Evaluating model with seed 666


C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

Evaluating model with seed 226688


C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\ProgramData\miniconda3\envs\raffael_env\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was ren

### Persistence

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# PERSISTENCE
seeds = [72, 88, 101, 666, 226688]
forecast_horizon = 48
seasonal_lag = 336 
stride = 48

# Generate test data
dm = BuildingEnergyDataModule(data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv", test_ratio=0.5)
dm.load_raw_data(data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv")
dm.setup()

_, _ = dl_to_sklearn_no_context(dm.train_dataloader())
_, y_test = dl_to_sklearn_no_context(dm.test_dataloader())

# Load target data
target_data = pd.read_csv("C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv").set_index("Unnamed: 0")['Site Total Energy (kWh)']
target_data.index = pd.to_datetime(target_data.index)


# Scale
scaler = MinMaxScaler()
scaler.fit(target_data.loc["2002"].values.reshape(-1, 1))
scaled_target_data = pd.Series(scaler.transform(target_data.values.reshape(-1, 1)).squeeze(), index=target_data.index)

# Vectorized extraction of the lagged forecast matrix
persistence_base = scaled_target_data.shift(seasonal_lag)
persistence_matrix = pd.concat([persistence_base.shift(-i) for i in range(forecast_horizon)], axis=1).dropna()

# Rearrange
y_test_hat_rs = rearrange(persistence_matrix[::stride].loc["2003-01-01":][int(seasonal_lag/forecast_horizon):].values, "B T -> (B T) 1")
y_test_rs = rearrange(y_test, "B T -> (B T) 1")

for feature_mode in ["endogen", "exogen"]:
    benchmark = []
    for seed in seeds:
        np.save(f"building/WeeklyNaive_{feature_mode}_{seed}_forecasts.npy", persistence_matrix[::stride].loc["2003-01-01":][int(seasonal_lag/forecast_horizon):].values)
        benchmark.append({"exp_name": f"WeeklyNaive_{feature_mode}",
                            "seed": seed,
                            "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(), 
                            "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs)**2).mean()})
    df = pd.DataFrame(benchmark).set_index(["exp_name", "seed"])
    df.to_csv(f"building/WeeklyNaive_{feature_mode}.csv")

# ARIMA

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.seasonal import MSTL
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (15,10)

target_data = pd.read_csv("C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv").set_index("Unnamed: 0")['Site Total Energy (kWh)']
target_data.index = pd.to_datetime(target_data.index)

#sm.graphics.tsa.plot_acf(target_data.values.squeeze(), lags=340)

res = MSTL(target_data.loc["2002"], periods=(48, 48*7)).fit()
res.plot()
residuals = res.resid
print("3")

In [ ]:
# Plot PACF for lag order
sm.graphics.tsa.plot_pacf(residual.diff().iloc[2:], lags=336)
plt.show()

In [ ]:
sm.graphics.tsa.plot_acf(residual.diff().iloc[2:], lags=336)
plt.show()

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string

df = target_data.loc["2002"].reset_index()
df["unique id"] = "1"
df.columns = ["ds", "y", "unique_id"]
df["ds"] = pd.to_datetime(df["ds"])

sf = StatsForecast(models=[AutoARIMA(max_p=336, max_d=3, max_q=336, max_P=336, max_D=3, max_Q=336)], freq='30min')
sf.fit(df=df)
arima_string(sf.fitted_[0,0].model_)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import seaborn as sns
import scipy.stats as stats

result = sf.fitted_[0,0].model_
print(result.keys())
print(result['arma'])

residual = pd.DataFrame(result.get("residuals"), columns=["residual Model"])
residual

fig, axs = plt.subplots(nrows=2, ncols=2)

# plot[1,1]
residual.plot(ax=axs[0,0])
axs[0,0].set_title("Residuals")

# plot
sns.distplot(residual, ax=axs[0,1])
axs[0,1].set_title("Density plot - Residual")

# plot
stats.probplot(residual["residual Model"], dist="norm", plot=axs[1,0])
axs[1,0].set_title('Plot Q-Q')

# plot
plot_acf(residual,  lags=336, ax=axs[1,1],color="fuchsia")
axs[1,1].set_title("Autocorrelation")

plt.show()

In [ ]:
target_data.loc["2002"]

# Collect results

In [ ]:
results = []
for name in ["XGBoost", "Catboost", "LinearRegression", "KNN", "MLP", "WeeklyNaive"]:
    for mode in ["endogen", "exogen"]:
        results.append(pd.read_csv(f"building/{name}_{mode}.csv"))
pd.concat(results).to_csv("all_benchmarks_building.csv")

In [ ]:
results = []
for name in ["LinearRegression", "KNN", "MLP", "WeeklyNaive"]:
    for mode in ["endogen", "exogen"]:
        results.append(pd.read_csv(f"railway/{name}_{mode}.csv"))
pd.concat(results).to_csv("all_benchmarks_railway.csv")